In [1]:
# ------------------------------------------
# NOTEBOOK DE CONSOLIDAÇÃO DE FEATURE ENGINEERING
# ------------------------------------------

In [2]:
# ------------------------------------------
# TAXA DE CONVERSAO BRL/USD : 4.40
# ------------------------------------------
PTAX = 4.40

In [3]:
# ------------------------------------------
# modulos pip
# ------------------------------------------
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

In [4]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [5]:
# ------------------------------------------
# DATABASE: historico dos couriers churneados pela plataforma
#
# SK.CREATED_AT::DATE : data de cadastro do courier na plataforma
# FECHA_ULT : ultima data de acesso do app pelo courier
#
# NOTA: esse database foi recebido com linhas duplicadas
# apos tratamento, total de linhas caiu de 32 milhões para 152,2 mil
# ------------------------------------------
df_churn = pd.read_csv("/content/drive/Shareddrives/grupo4-rappi-hour/bases-rappi/criacao-contas-churn-sem-duplicados-prov.csv")
df_churn.drop(labels="Unnamed: 0", axis=1, inplace = True)
print(f"Total de linhas: {df_churn.shape[0]}")
df_churn.head(3)

Total de linhas: 152224


,ID,FIRST_NAME,GENDER,CITY,SK.CREATED_AT::DATE,TRANSPORT_MEDIA_TYPE,CARTAO,LEVEL_NAME,FECHA_ULT
0,1286316,Adailton,M,Grande São Paulo,2021-06-07,motorbike,True,bronze,2022-01-16T23:27:35Z
1,1110698,Adriano Floriano Da Silva,M,Recife,2021-02-11,bicycle,True,bronze,2021-07-15T11:16:04Z
2,284886,Bruno,M,Grande São Paulo,2019-07-03,motorbike,False,bronze,2021-07-07T12:33:21Z


In [6]:
# ------------------------------------------
# database contem dados de cidades estrangeiras e nulos
# EXEMPLO : Tuxtla Gutierrez [MX] ; without_city
# ------------------------------------------
df_churn[df_churn['CITY'].isin([ "Tuxtla Gutierrez", "without_city" ])].head()

,ID,FIRST_NAME,GENDER,CITY,SK.CREATED_AT::DATE,TRANSPORT_MEDIA_TYPE,CARTAO,LEVEL_NAME,FECHA_ULT
8727,429842,Marcilon Antônio,M,without_city,2019-10-19,bicycle,True,rookie,2021-09-09T13:48:10Z
10994,1310230,Carla,F,without_city,2021-07-02,motorbike,False,bronze,2021-08-30T15:33:51Z
11318,1127727,Leandro,M,without_city,2021-02-21,bicycle,True,bronze,2021-09-09T13:04:48Z
12060,584838,Wadson,M,Tuxtla Gutierrez,2020-02-27,motorbike,True,bronze,2021-08-06T22:54:12Z
21241,547765,Thyago,M,without_city,2020-01-27,motorbike,True,bronze,2021-09-09T17:50:33Z


In [7]:
# ------------------------------------------
# database contem couriers que abandonaram a plataforma
# entre junho de 2021 a julho de 2022
# ------------------------------------------
df_churn["FECHA_ULT"].map(lambda x : x[:7]).value_counts().sort_index()

2021-06    11633
2021-07    12387
2021-08    11777
2021-09     9838
2021-10     9760
2021-11     8653
2021-12     8374
2022-01     9337
2022-02     9966
2022-03    13781
2022-04    13602
2022-05    15140
2022-06    16030
2022-07     1946
Name: FECHA_ULT, dtype: int64

In [8]:
# ------------------------------------------
# 80% do churn está concentrado nas seguintes cidades
#     . Grande São Paulo
#     . Rio de Janeiro
#     . Belo Horizonte
#     . Recife
#     . Curitiba
#     . Fortaleza
#     . Porto Alegre
#
# DECISAO : trabalhar apenas com couriers que atuem nessas cidades
# ------------------------------------------
df_conc_churn = (df_churn["CITY"].value_counts(normalize=True).cumsum() < 0.8).to_frame()
df_conc_by_city = df_conc_churn[df_conc_churn["CITY"] == True]
df_conc_by_city.index

Index(['Grande São Paulo', 'Rio de Janeiro', 'Belo Horizonte', 'Recife',
       'Curitiba', 'Fortaleza', 'Porto Alegre'],
      dtype='object')

In [9]:
# ------------------------------------------
# CONCLUSAO - FEATURE ENGINEERING - DATABASES:
#     . CRIACAO CONTAS CHURN : filtragem dos couriers que atuam nos principais
#       mercados
# ------------------------------------------
filter = df_conc_by_city.index.to_list()
df_churn_conc_by_city = df_churn[df_churn['CITY'].isin(filter)]
print(f"Total de linhas: {df_churn_conc_by_city.shape[0]}")
df_churn_conc_by_city.head(3)

Total de linhas: 120206


,ID,FIRST_NAME,GENDER,CITY,SK.CREATED_AT::DATE,TRANSPORT_MEDIA_TYPE,CARTAO,LEVEL_NAME,FECHA_ULT
0,1286316,Adailton,M,Grande São Paulo,2021-06-07,motorbike,True,bronze,2022-01-16T23:27:35Z
1,1110698,Adriano Floriano Da Silva,M,Recife,2021-02-11,bicycle,True,bronze,2021-07-15T11:16:04Z
2,284886,Bruno,M,Grande São Paulo,2019-07-03,motorbike,False,bronze,2021-07-07T12:33:21Z


In [10]:
# ------------------------------------------
# DATABASE : informacoes gerais de couriers ativos na plataforma
#
# is_active (bool) : true se ativo ; false se inativo
# ------------------------------------------
df_infos_gerais = pd.read_csv("/content/drive/Shareddrives/grupo4-rappi-hour/bases-rappi/infos-gerais-prov.csv")
print(f"Total de linhas: {df_infos_gerais.shape[0]}")
perc_ativos = round(df_infos_gerais["IS_ACTIVE"].value_counts(normalize=True)[1] * 100, 1)
print(f"% de couriers ativos no database: {perc_ativos}%")
df_infos_gerais.head(3)

Total de linhas: 180178
% de couriers ativos no database: 99.8%


,ID,NOME,SOBRENOME,GENERO,DATA_NASCIMENTO,CIDADE,IS_ACTIVE,TRANSPORTE,AUTO_ACEITE,COUNT_ORDERS_LAST_7D,...,ULTIMO_PEDIDO,COUNT_ORDERS_RESTAURANTES,COUNT_ORDERS_MERCADO,COUNT_ORDERS_FARMACIA,COUNT_ORDERS_EXPRESS,COUNT_ORDERS_ECOMMERCE,COUNT_ORDERS_ANTOJO,FRETE_MEDIO,COOKING_TIME_MEDIO,ITENS_MEDIO
0,1561246,Wilton Jhonne,Da Silva Abreu,M,1988-04-21,Sao Paulo,True,motorbike,True,1,...,2022-08-01T00:00:00Z,1,0,0,0,0,0,62.255500,10.000000,1.0
1,1561210,Dennis Leonardo,Pereira Santos,M,1998-06-28,Grande São Paulo,True,motorbike,True,7,...,2022-08-01T00:00:00Z,6,1,0,0,0,0,43.444714,23.142857,3.0
2,1561205,Thavillo Teles,Balviano De Olivetra,M,1997-09-25,Natal,True,motorbike,True,2,...,2022-08-01T00:00:00Z,1,1,0,0,0,0,42.230500,4.500000,3.5


In [11]:
# ------------------------------------------
# padronizacao de grafia das cidades usadas para
# filtragem dos couriers
# ------------------------------------------
df_infos_gerais.replace(
    {"CIDADE": {
        'Sao Paulo': 'Grande São Paulo',
        'São Paulo': 'Grande São Paulo',
        'SÃO PAULO': 'Grande São Paulo',
        'BELO HORIZINTE': 'Belo Horizonte',
        'RIO DE JANEIRO': 'Rio de Janeiro'
    }}, inplace=True
)

In [12]:
# ------------------------------------------
# filtragem dos couriers ativos que atuam nos principais mercados 
# ------------------------------------------
df_infos_gerais_conc_by_city = df_infos_gerais[df_infos_gerais['CIDADE'].isin(filter)]
df_infos_gerais_conc_by_city["CIDADE"].unique()

array(['Grande São Paulo', 'Recife', 'Fortaleza', 'Belo Horizonte',
       'Rio de Janeiro', 'Porto Alegre', 'Curitiba'], dtype=object)

In [13]:
# ------------------------------------------
# cruzamento dos couriers inativos de acordo com o database de churn
# e os dados gerais de couriers ativos do database de infos gerais
# 
# RESULTADO ESPERADO : baixa repeticao de IDs entre os databases
# RESULTADO OBSERVADO : alta repeticao de IDs (101.140)
#
# CONCLUSAO : repeticao elevada reflete couriers que abandonam a
# plataforma periodicamente e acabam retornando apos um periodo
# ------------------------------------------
filter = df_churn_conc_by_city["ID"].to_list()
df_infos_gerais_conc_by_city[df_infos_gerais_conc_by_city["ID"].isin(filter)].shape[0]

101140

In [14]:
# ------------------------------------------
# EXEMPLO (part 1 de 2):
# id 1544047 - Raquel da Costa Mendes, consta como churn, tendo
# acessado a plataforma pela ultima vez em 03-07-2022
# ------------------------------------------
print(df_churn_conc_by_city[df_churn_conc_by_city["ID"] == 1544047])

            ID FIRST_NAME GENDER            CITY SK.CREATED_AT::DATE  \
44151  1544047     Raquel      F  Belo Horizonte          2022-07-03   

      TRANSPORT_MEDIA_TYPE CARTAO LEVEL_NAME             FECHA_ULT  
44151                  car  False     rookie  2022-07-03T20:29:39Z  


In [15]:
# ------------------------------------------
# EXEMPLO (part 2 de 2):
# id 1544047 - Raquel da Costa Mendes, consta com status ATIVO
# ------------------------------------------
print(df_infos_gerais_conc_by_city[df_infos_gerais_conc_by_city["ID"] == 1544047])

           ID    NOME        SOBRENOME GENERO DATA_NASCIMENTO          CIDADE  \
3572  1544047  Raquel  Da Costa Mendes      F      1983-03-01  Belo Horizonte   

      IS_ACTIVE TRANSPORTE  AUTO_ACEITE  COUNT_ORDERS_LAST_7D  ...  \
3572       True        car         True                     0  ...   

             ULTIMO_PEDIDO  COUNT_ORDERS_RESTAURANTES  COUNT_ORDERS_MERCADO  \
3572  2022-07-03T00:00:00Z                          0                     0   

      COUNT_ORDERS_FARMACIA COUNT_ORDERS_EXPRESS COUNT_ORDERS_ECOMMERCE  \
3572                      0                    0                      0   

      COUNT_ORDERS_ANTOJO  FRETE_MEDIO  COOKING_TIME_MEDIO  ITENS_MEDIO  
3572                    0      53.3555                 NaN          2.0  

[1 rows x 25 columns]


In [16]:
# ------------------------------------------
# CONCLUSAO - FEATURE ENGINEERING - DATABASES:
#     . CRIACAO CONTAS CHURN
#     . INFOS GERAIS 
#
# DECISAO : tierizar os dados de churn (col: IS_ACTIVE)
#     . categoria 1 : True - couriers que nunca foram churneados
#     . categoria 2 : Quasi - couriers inconsistentes
#     . categoria 3 : False - couriers com churn definitivo
# ------------------------------------------
df_infos_gerais_conc_by_city.head(2)

,ID,NOME,SOBRENOME,GENERO,DATA_NASCIMENTO,CIDADE,IS_ACTIVE,TRANSPORTE,AUTO_ACEITE,COUNT_ORDERS_LAST_7D,...,ULTIMO_PEDIDO,COUNT_ORDERS_RESTAURANTES,COUNT_ORDERS_MERCADO,COUNT_ORDERS_FARMACIA,COUNT_ORDERS_EXPRESS,COUNT_ORDERS_ECOMMERCE,COUNT_ORDERS_ANTOJO,FRETE_MEDIO,COOKING_TIME_MEDIO,ITENS_MEDIO
0,1561246,Wilton Jhonne,Da Silva Abreu,M,1988-04-21,Grande São Paulo,True,motorbike,True,1,...,2022-08-01T00:00:00Z,1,0,0,0,0,0,62.255500,10.000000,1.0
1,1561210,Dennis Leonardo,Pereira Santos,M,1998-06-28,Grande São Paulo,True,motorbike,True,7,...,2022-08-01T00:00:00Z,6,1,0,0,0,0,43.444714,23.142857,3.0


In [17]:
# ------------------------------------------
# filter : lista dos couriers flagados como churn no database CRIACAO CONTAS CHURN
# mask : checa se os couriers flagados constam como ativos no database INFOS GERAIS
# CONCLUSAO : se houver match, o courier é inconsistente e categorizado como Quasi
# ------------------------------------------
filter = df_churn_conc_by_city["ID"].to_list()
mask = df_infos_gerais_conc_by_city["ID"].isin(filter)
column_name = "IS_ACTIVE"
df_infos_gerais_conc_by_city.loc[mask, column_name] = "Quasi"

/usr/local/lib/python3.7/dist-packages/pandas/core/indexing.py:1817: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  self._setitem_single_column(loc, value, pi)


In [18]:
# ------------------------------------------
# checagem da distribuicao de categorias apos aplicacao do filtro
# ------------------------------------------
df_infos_gerais_conc_by_city["IS_ACTIVE"].value_counts(normalize=True)

Quasi    0.719806
True     0.279311
False    0.000882
Name: IS_ACTIVE, dtype: float64

In [19]:
# ------------------------------------------
# MERGE DOS DATABASES : CRIACAO CONTAS CHURN *AND* INFOS GERAIS
# OBJETIVO : incluir couriers com churn definitivo no database de consolidacao
# ------------------------------------------

In [20]:
# ------------------------------------------
# visualização das dimensões e colunas do database de churn
# ------------------------------------------
print(f"Dimensões do database de churn: {df_churn_conc_by_city.shape}")
df_churn_conc_by_city.head(2)

Dimensões do database de churn: (120206, 9)


,ID,FIRST_NAME,GENDER,CITY,SK.CREATED_AT::DATE,TRANSPORT_MEDIA_TYPE,CARTAO,LEVEL_NAME,FECHA_ULT
0,1286316,Adailton,M,Grande São Paulo,2021-06-07,motorbike,True,bronze,2022-01-16T23:27:35Z
1,1110698,Adriano Floriano Da Silva,M,Recife,2021-02-11,bicycle,True,bronze,2021-07-15T11:16:04Z


In [21]:
# ------------------------------------------
# visualização das dimensões e colunas do database de infos gerais
# ------------------------------------------
print(f"Dimensões do database de infos gerais: {df_infos_gerais_conc_by_city.shape}")
df_infos_gerais_conc_by_city.head(2)

Dimensões do database de infos gerais: (140510, 25)


,ID,NOME,SOBRENOME,GENERO,DATA_NASCIMENTO,CIDADE,IS_ACTIVE,TRANSPORTE,AUTO_ACEITE,COUNT_ORDERS_LAST_7D,...,ULTIMO_PEDIDO,COUNT_ORDERS_RESTAURANTES,COUNT_ORDERS_MERCADO,COUNT_ORDERS_FARMACIA,COUNT_ORDERS_EXPRESS,COUNT_ORDERS_ECOMMERCE,COUNT_ORDERS_ANTOJO,FRETE_MEDIO,COOKING_TIME_MEDIO,ITENS_MEDIO
0,1561246,Wilton Jhonne,Da Silva Abreu,M,1988-04-21,Grande São Paulo,True,motorbike,True,1,...,2022-08-01T00:00:00Z,1,0,0,0,0,0,62.255500,10.000000,1.0
1,1561210,Dennis Leonardo,Pereira Santos,M,1998-06-28,Grande São Paulo,True,motorbike,True,7,...,2022-08-01T00:00:00Z,6,1,0,0,0,0,43.444714,23.142857,3.0


In [22]:
# ------------------------------------------
# filter : lista dos IDs dos couriers ativos nas principais cidades
# mask : checa os couriers que constam exclusivamente da base de churn 
# df_churn_def : dataframe com dados apenas dos couriers que churnearam a
# plataforma de modo definitivo
# ------------------------------------------
filter = df_infos_gerais_conc_by_city["ID"].to_list()
mask = ~df_churn_conc_by_city["ID"].isin(filter)
df_churn_def = df_churn_conc_by_city[mask]
df_churn_def.head(2)

,ID,FIRST_NAME,GENDER,CITY,SK.CREATED_AT::DATE,TRANSPORT_MEDIA_TYPE,CARTAO,LEVEL_NAME,FECHA_ULT
1,1110698,Adriano Floriano Da Silva,M,Recife,2021-02-11,bicycle,True,bronze,2021-07-15T11:16:04Z
2,284886,Bruno,M,Grande São Paulo,2019-07-03,motorbike,False,bronze,2021-07-07T12:33:21Z


In [23]:
# ------------------------------------------
# padronizacao dos nomes das colunas e drop das colunas consideradas
# dispensáveis para a construção do modelo
# ------------------------------------------
df_churn_def.rename(columns={
    "FIRST_NAME": "NOME", 
    "GENDER": "GENERO",
    "CITY": "CIDADE",
    "TRANSPORT_MEDIA_TYPE": "TRANSPORTE"
    }, inplace=True)

df_churn_def.drop(columns=["SK.CREATED_AT::DATE", "CARTAO", "FECHA_ULT", "LEVEL_NAME"], inplace=True)
df_churn_def["IS_ACTIVE"] = False

/usr/local/lib/python3.7/dist-packages/pandas/core/frame.py:5047: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  errors=errors,
/usr/local/lib/python3.7/dist-packages/pandas/core/frame.py:4913: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  errors=errors,
/usr/local/lib/python3.7/dist-packages/ipykernel_launcher.py:13: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  del sys.path[0]


In [24]:
# ------------------------------------------
# concatenação das bases para consolidação de informações de couriers
# segmentos entre ativos (IS_ACTIVE == True), inativos (IS_ACTIVE == False) e
# inconsistentes (IS_ACTIVE == Quasi)
# ------------------------------------------
df_consol = pd.concat([df_infos_gerais_conc_by_city, df_churn_def])
df_consol.head(2)

,ID,NOME,SOBRENOME,GENERO,DATA_NASCIMENTO,CIDADE,IS_ACTIVE,TRANSPORTE,AUTO_ACEITE,COUNT_ORDERS_LAST_7D,...,ULTIMO_PEDIDO,COUNT_ORDERS_RESTAURANTES,COUNT_ORDERS_MERCADO,COUNT_ORDERS_FARMACIA,COUNT_ORDERS_EXPRESS,COUNT_ORDERS_ECOMMERCE,COUNT_ORDERS_ANTOJO,FRETE_MEDIO,COOKING_TIME_MEDIO,ITENS_MEDIO
0,1561246,Wilton Jhonne,Da Silva Abreu,M,1988-04-21,Grande São Paulo,True,motorbike,True,1.0,...,2022-08-01T00:00:00Z,1.0,0.0,0.0,0.0,0.0,0.0,62.255500,10.000000,1.0
1,1561210,Dennis Leonardo,Pereira Santos,M,1998-06-28,Grande São Paulo,True,motorbike,True,7.0,...,2022-08-01T00:00:00Z,6.0,1.0,0.0,0.0,0.0,0.0,43.444714,23.142857,3.0


In [25]:
# ------------------------------------------
# Distribuicao final das categorias de couriers
#
# A partir dessa amostra, poderemos selecionar features e rodar modelos
# com objetivo de melhor categorizar os couriers com base em atributos observaveis
# ------------------------------------------
df_consol["IS_ACTIVE"].value_counts(normalize=True)

Quasi    0.633805
True     0.245939
False    0.120256
Name: IS_ACTIVE, dtype: float64

In [26]:
# ------------------------------------------
# DETALHES SOBRE COMPOSIÇÃO DA BASE CONSOLIDADA
# ------------------------------------------
df_consol.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 159576 entries, 0 to 152223
Data columns (total 25 columns):
 #   Column                          Non-Null Count   Dtype  
---  ------                          --------------   -----  
 0   ID                              159576 non-null  int64  
 1   NOME                            159562 non-null  object 
 2   SOBRENOME                       140501 non-null  object 
 3   GENERO                          159576 non-null  object 
 4   DATA_NASCIMENTO                 140510 non-null  object 
 5   CIDADE                          159576 non-null  object 
 6   IS_ACTIVE                       159576 non-null  object 
 7   TRANSPORTE                      159576 non-null  object 
 8   AUTO_ACEITE                     140510 non-null  object 
 9   COUNT_ORDERS_LAST_7D            140510 non-null  float64
 10  COUNT_ORDERS_LAST_30D           140510 non-null  float64
 11  COUNT_ORDERS_CANCELED_LAST_7D   140510 non-null  float64
 12  COUNT_ORDERS_CAN

In [27]:
# ------------------------------------------
# drop de colunas consideradas não relevantes (nome e sobrenome)
# conversão de data de nascimento (object) para ano de nascimento (int)
# ------------------------------------------
df_consol.drop(columns=["NOME", "SOBRENOME"], inplace=True)
df_consol.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 159576 entries, 0 to 152223
Data columns (total 23 columns):
 #   Column                          Non-Null Count   Dtype  
---  ------                          --------------   -----  
 0   ID                              159576 non-null  int64  
 1   GENERO                          159576 non-null  object 
 2   DATA_NASCIMENTO                 140510 non-null  object 
 3   CIDADE                          159576 non-null  object 
 4   IS_ACTIVE                       159576 non-null  object 
 5   TRANSPORTE                      159576 non-null  object 
 6   AUTO_ACEITE                     140510 non-null  object 
 7   COUNT_ORDERS_LAST_7D            140510 non-null  float64
 8   COUNT_ORDERS_LAST_30D           140510 non-null  float64
 9   COUNT_ORDERS_CANCELED_LAST_7D   140510 non-null  float64
 10  COUNT_ORDERS_CANCELED_LAST_30D  140510 non-null  float64
 11  GORJETA                         140510 non-null  float64
 12  PRIMEIRO_PEDIDO 

In [28]:
# ------------------------------------------
# tratamento e conversão da data de nascimento para numero
# ------------------------------------------
df_consol["DATA_NASCIMENTO"] = df_consol["DATA_NASCIMENTO"].map(lambda x : x[:4], na_action="ignore")
df_consol["DATA_NASCIMENTO"] = pd.to_numeric(df_consol["DATA_NASCIMENTO"])
mean_avg_birthyear = round(df_consol["DATA_NASCIMENTO"].mean(), 0)
df_consol["DATA_NASCIMENTO"].fillna(mean_avg_birthyear, inplace=True)
df_consol.head(2)

,ID,GENERO,DATA_NASCIMENTO,CIDADE,IS_ACTIVE,TRANSPORTE,AUTO_ACEITE,COUNT_ORDERS_LAST_7D,COUNT_ORDERS_LAST_30D,COUNT_ORDERS_CANCELED_LAST_7D,...,ULTIMO_PEDIDO,COUNT_ORDERS_RESTAURANTES,COUNT_ORDERS_MERCADO,COUNT_ORDERS_FARMACIA,COUNT_ORDERS_EXPRESS,COUNT_ORDERS_ECOMMERCE,COUNT_ORDERS_ANTOJO,FRETE_MEDIO,COOKING_TIME_MEDIO,ITENS_MEDIO
0,1561246,M,1988.0,Grande São Paulo,True,motorbike,True,1.0,1.0,0.0,...,2022-08-01T00:00:00Z,1.0,0.0,0.0,0.0,0.0,0.0,62.255500,10.000000,1.0
1,1561210,M,1998.0,Grande São Paulo,True,motorbike,True,7.0,7.0,0.0,...,2022-08-01T00:00:00Z,6.0,1.0,0.0,0.0,0.0,0.0,43.444714,23.142857,3.0


In [29]:
# ------------------------------------------
# teste de colunas com dados faltantes devido a
# importacao de IDs churneados da base de dados de churn
# ------------------------------------------
if (
    (df_consol[df_consol["COUNT_ORDERS_LAST_7D"].isna()]["IS_ACTIVE"].unique() == False) & 
    (df_consol[df_consol["COUNT_ORDERS_LAST_30D"].isna()]["IS_ACTIVE"].unique() == False) &
    (df_consol[df_consol["COUNT_ORDERS_CANCELED_LAST_7D"].isna()]["IS_ACTIVE"].unique() == False) &
    (df_consol[df_consol["COUNT_ORDERS_CANCELED_LAST_30D"].isna()]["IS_ACTIVE"].unique() == False) &
    (df_consol[df_consol["GORJETA"].isna()]["IS_ACTIVE"].unique() == False) &
    (df_consol[df_consol["COUNT_ORDERS_RESTAURANTES"].isna()]["IS_ACTIVE"].unique() == False) &
    (df_consol[df_consol["COUNT_ORDERS_MERCADO"].isna()]["IS_ACTIVE"].unique() == False) &
    (df_consol[df_consol["COUNT_ORDERS_FARMACIA"].isna()]["IS_ACTIVE"].unique() == False) &
    (df_consol[df_consol["COUNT_ORDERS_EXPRESS"].isna()]["IS_ACTIVE"].unique() == False) &
    (df_consol[df_consol["COUNT_ORDERS_ANTOJO"].isna()]["IS_ACTIVE"].unique() == False)
    ):
  print("Informações faltantes são apenas de churns definitivos")

Informações faltantes são apenas de churns definitivos


In [30]:
# ------------------------------------------
# confirmada a hipotese acima, os dados podem
# sem completados com 0 sem prejuizo a analise
# ------------------------------------------
df_consol["COUNT_ORDERS_LAST_7D"].fillna(0, inplace=True)
df_consol["COUNT_ORDERS_LAST_30D"].fillna(0, inplace=True)
df_consol["COUNT_ORDERS_CANCELED_LAST_7D"].fillna(0, inplace=True)
df_consol["COUNT_ORDERS_CANCELED_LAST_30D"].fillna(0, inplace=True)
df_consol["GORJETA"].fillna(0, inplace=True)
df_consol["COUNT_ORDERS_RESTAURANTES"].fillna(0, inplace=True)
df_consol["COUNT_ORDERS_MERCADO"].fillna(0, inplace=True)
df_consol["COUNT_ORDERS_FARMACIA"].fillna(0, inplace=True)
df_consol["COUNT_ORDERS_EXPRESS"].fillna(0, inplace=True)
df_consol["COUNT_ORDERS_ANTOJO"].fillna(0, inplace=True)

In [31]:
df_consol.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 159576 entries, 0 to 152223
Data columns (total 23 columns):
 #   Column                          Non-Null Count   Dtype  
---  ------                          --------------   -----  
 0   ID                              159576 non-null  int64  
 1   GENERO                          159576 non-null  object 
 2   DATA_NASCIMENTO                 159576 non-null  float64
 3   CIDADE                          159576 non-null  object 
 4   IS_ACTIVE                       159576 non-null  object 
 5   TRANSPORTE                      159576 non-null  object 
 6   AUTO_ACEITE                     140510 non-null  object 
 7   COUNT_ORDERS_LAST_7D            159576 non-null  float64
 8   COUNT_ORDERS_LAST_30D           159576 non-null  float64
 9   COUNT_ORDERS_CANCELED_LAST_7D   159576 non-null  float64
 10  COUNT_ORDERS_CANCELED_LAST_30D  159576 non-null  float64
 11  GORJETA                         159576 non-null  float64
 12  PRIMEIRO_PEDIDO 

In [32]:
# ------------------------------------------
# DATABASE: ????? TODO : descrição da tabela
# ------------------------------------------
df_distance_user = pd.read_csv("/content/drive/Shareddrives/grupo4-rappi-hour/bases-rappi/distance-user-def.csv")
print(f"Total de linhas: {df_distance_user.shape[0]}")
df_distance_user.head(3)

/usr/local/lib/python3.7/dist-packages/IPython/core/interactiveshell.py:3326: DtypeWarning: Columns (3) have mixed types.Specify dtype option on import or set low_memory=False.
  exec(code_obj, self.user_global_ns, self.user_ns)


Total de linhas: 31382215


,ORDER_ID,STOREKEEPER_ID,DISTANCE_TO_USER,BUNDLE_ID
0,117818357,NaN,2.315865,NaN
1,147728554,NaN,2.584614,NaN
2,147959308,NaN,3.641370,NaN


In [33]:
df_distance_user[~df_distance_user["STOREKEEPER_ID"].isna()]

,ORDER_ID,STOREKEEPER_ID,DISTANCE_TO_USER,BUNDLE_ID
299493,150240162,74280.0,2.669719,d2809ab7-64e2-4faf-bf23-e11e0ea07a68
299494,150238346,477768.0,2.620474,5330dbea-69c4-467b-b743-c5d58a0a94c0
299495,150240000,184461.0,3.934305,b861b992-3165-4db7-b248-df764a8ac682
299496,150239077,1318848.0,2.106841,5c39e64a-d842-4406-a9f0-9d6091215ae0
299497,150238345,1157640.0,5.319865,684271b3-4f1f-4f1c-9aff-e923a6a13bd7
...,...,...,...,...
31382210,128962444,1208502.0,0.499839,NaN
31382211,107665980,406117.0,2.398273,NaN
31382212,110061059,239773.0,0.997030,NaN
31382213,135040280,1223363.0,1.222013,NaN


In [34]:
df_distance_user_grouped = df_distance_user.groupby(["STOREKEEPER_ID"]).mean()
df_distance_user_grouped.drop(["ORDER_ID"], axis=1, inplace=True)

In [35]:
df_distance_user_grouped = df_distance_user_grouped.reset_index()
df_distance_user_grouped.rename(columns={
    "STOREKEEPER_ID": "ID"
    }, inplace=True)

In [36]:
df_distance_user_grouped.head(2)

,ID,DISTANCE_TO_USER
0,32818.0,1.896183
1,33051.0,1.550505


In [37]:
df_consol

,ID,GENERO,DATA_NASCIMENTO,CIDADE,IS_ACTIVE,TRANSPORTE,AUTO_ACEITE,COUNT_ORDERS_LAST_7D,COUNT_ORDERS_LAST_30D,COUNT_ORDERS_CANCELED_LAST_7D,...,ULTIMO_PEDIDO,COUNT_ORDERS_RESTAURANTES,COUNT_ORDERS_MERCADO,COUNT_ORDERS_FARMACIA,COUNT_ORDERS_EXPRESS,COUNT_ORDERS_ECOMMERCE,COUNT_ORDERS_ANTOJO,FRETE_MEDIO,COOKING_TIME_MEDIO,ITENS_MEDIO
0,1561246,M,1988.0,Grande São Paulo,True,motorbike,True,1.0,1.0,0.0,...,2022-08-01T00:00:00Z,1.0,0.0,0.0,0.0,0.0,0.0,62.255500,10.000000,1.00
1,1561210,M,1998.0,Grande São Paulo,True,motorbike,True,7.0,7.0,0.0,...,2022-08-01T00:00:00Z,6.0,1.0,0.0,0.0,0.0,0.0,43.444714,23.142857,3.00
3,1561173,F,1994.0,Grande São Paulo,True,motorbike,True,4.0,4.0,0.0,...,2022-08-01T00:00:00Z,4.0,0.0,0.0,0.0,0.0,0.0,47.236750,7.250000,1.25
5,1561061,M,1993.0,Recife,True,motorbike,True,2.0,2.0,0.0,...,2022-08-01T00:00:00Z,1.0,0.0,0.0,0.0,0.0,0.0,35.555500,10.000000,2.50
6,1561019,F,2004.0,Fortaleza,True,bicycle,True,0.0,0.0,1.0,...,2022-08-01T00:00:00Z,0.0,0.0,0.0,0.0,0.0,0.0,57.805500,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
152216,1502887,M,1990.0,Curitiba,False,motorbike,NaN,0.0,0.0,0.0,...,NaN,0.0,0.0,0.0,0.0,NaN,0.0,NaN,NaN,NaN
152217,1493340,M,1990.0,Belo Horizonte,False,motorbike,NaN,0.0,0.0,0.0,...,NaN,0.0,0.0,0.0,0.0,NaN,0.0,NaN,NaN,NaN
152220,1503585,M,1990.0,Curitiba,False,bicycle,NaN,0.0,0.0,0.0,...,NaN,0.0,0.0,0.0,0.0,NaN,0.0,NaN,NaN,NaN
152221,1399642,M,1990.0,Grande São Paulo,False,motorbike,NaN,0.0,0.0,0.0,...,NaN,0.0,0.0,0.0,0.0,NaN,0.0,NaN,NaN,NaN


In [38]:
df_consol = pd.merge(df_consol, df_distance_user_grouped, how="left", on="ID")
df_consol.head(2)

,ID,GENERO,DATA_NASCIMENTO,CIDADE,IS_ACTIVE,TRANSPORTE,AUTO_ACEITE,COUNT_ORDERS_LAST_7D,COUNT_ORDERS_LAST_30D,COUNT_ORDERS_CANCELED_LAST_7D,...,COUNT_ORDERS_RESTAURANTES,COUNT_ORDERS_MERCADO,COUNT_ORDERS_FARMACIA,COUNT_ORDERS_EXPRESS,COUNT_ORDERS_ECOMMERCE,COUNT_ORDERS_ANTOJO,FRETE_MEDIO,COOKING_TIME_MEDIO,ITENS_MEDIO,DISTANCE_TO_USER
0,1561246,M,1988.0,Grande São Paulo,True,motorbike,True,1.0,1.0,0.0,...,1.0,0.0,0.0,0.0,0.0,0.0,62.255500,10.000000,1.0,3.161503
1,1561210,M,1998.0,Grande São Paulo,True,motorbike,True,7.0,7.0,0.0,...,6.0,1.0,0.0,0.0,0.0,0.0,43.444714,23.142857,3.0,4.079120


In [39]:
df_consol["DISTANCE_TO_USER"].isna().sum()

10791

In [40]:
# ------------------------------------------
# DATABASE: 
# ------------------------------------------
df_orders_done_cancel = pd.read_csv("/content/drive/Shareddrives/grupo4-rappi-hour/bases-rappi/ordens-done-cancel-def.csv")
print(f"Total de linhas: {df_orders_done_cancel.shape[0]}")
df_orders_done_cancel.head(3)

Total de linhas: 653166


,STOREKEEPER_ID,ORDERS_DONE,ORDERS_CANCEL,CANCELS_OPS_RT
0,266155,10356,61,0
1,166971,10272,112,4
2,2144,9915,123,0


In [41]:
df_orders_done_cancel = df_orders_done_cancel.groupby(["STOREKEEPER_ID"]).mean()
df_orders_done_cancel

,ORDERS_DONE,ORDERS_CANCEL,CANCELS_OPS_RT
STOREKEEPER_ID,,,
11,0.0,16.0,0.0
12,82.0,0.0,0.0
14,2.0,0.0,0.0
19,3075.0,26.0,0.0
23,60.0,3.0,0.0
...,...,...,...
10002501,29.0,0.0,0.0
10002502,3.0,2.0,0.0
10002503,33.0,1.0,0.0


In [42]:
df_orders_done_cancel = df_orders_done_cancel.reset_index()
df_orders_done_cancel.rename(columns={
    "STOREKEEPER_ID": "ID"
    }, inplace=True)
df_orders_done_cancel

,ID,ORDERS_DONE,ORDERS_CANCEL,CANCELS_OPS_RT
0,11,0.0,16.0,0.0
1,12,82.0,0.0,0.0
2,14,2.0,0.0,0.0
3,19,3075.0,26.0,0.0
4,23,60.0,3.0,0.0
...,...,...,...,...
653161,10002501,29.0,0.0,0.0
653162,10002502,3.0,2.0,0.0
653163,10002503,33.0,1.0,0.0
653164,10002504,18.0,0.0,0.0


In [43]:
df_consol = pd.merge(df_consol, df_orders_done_cancel, how="left", on="ID")
df_consol.head(2)

,ID,GENERO,DATA_NASCIMENTO,CIDADE,IS_ACTIVE,TRANSPORTE,AUTO_ACEITE,COUNT_ORDERS_LAST_7D,COUNT_ORDERS_LAST_30D,COUNT_ORDERS_CANCELED_LAST_7D,...,COUNT_ORDERS_EXPRESS,COUNT_ORDERS_ECOMMERCE,COUNT_ORDERS_ANTOJO,FRETE_MEDIO,COOKING_TIME_MEDIO,ITENS_MEDIO,DISTANCE_TO_USER,ORDERS_DONE,ORDERS_CANCEL,CANCELS_OPS_RT
0,1561246,M,1988.0,Grande São Paulo,True,motorbike,True,1.0,1.0,0.0,...,0.0,0.0,0.0,62.255500,10.000000,1.0,3.161503,NaN,NaN,NaN
1,1561210,M,1998.0,Grande São Paulo,True,motorbike,True,7.0,7.0,0.0,...,0.0,0.0,0.0,43.444714,23.142857,3.0,4.079120,NaN,NaN,NaN


In [47]:
df_consol.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 159576 entries, 0 to 159575
Data columns (total 27 columns):
 #   Column                          Non-Null Count   Dtype  
---  ------                          --------------   -----  
 0   ID                              159576 non-null  int64  
 1   GENERO                          159576 non-null  object 
 2   DATA_NASCIMENTO                 159576 non-null  float64
 3   CIDADE                          159576 non-null  object 
 4   IS_ACTIVE                       159576 non-null  object 
 5   TRANSPORTE                      159576 non-null  object 
 6   AUTO_ACEITE                     140510 non-null  object 
 7   COUNT_ORDERS_LAST_7D            159576 non-null  float64
 8   COUNT_ORDERS_LAST_30D           159576 non-null  float64
 9   COUNT_ORDERS_CANCELED_LAST_7D   159576 non-null  float64
 10  COUNT_ORDERS_CANCELED_LAST_30D  159576 non-null  float64
 11  GORJETA                         159576 non-null  float64
 12  PRIMEIRO_PEDIDO 

In [48]:
# ------------------------------------------
# DATABASE: sempre que um usuário abre alguma reclamação
# sobre algum pedido incompleto, faltante, item errado,
# é gerada uma ORDER_DEFECT
#
# ORDERS : numero de pedidos que aconteceu algum problema
# GMV_TOTAL : Total da transação (GMV = custo total pago pela Rappi para a loja)
# ------------------------------------------
df_comp_defects = pd.read_csv("/content/drive/Shareddrives/grupo4-rappi-hour/bases-rappi/comp-defects-def.csv")
print(f"Total de linhas: {df_comp_defects.shape[0]}")
df_comp_defects.head(3)

Total de linhas: 6783958


,STOREKEEPER_ID,WEEK,CITY,LEVEL_ID,LEVEL_NAME,ORDERS,GMV_TOTAL,COMPENSATIONS,DEFECT_COMPENSATIONS,DEFECT_ORDER
0,1009854.0,2021-07-17,Grande São Paulo,3.0,bronze,5,422.48,11.173,108019818.0,108019818.0
1,822496.0,2022-01-19,Grande São Paulo,3.0,bronze,5,947.92,1.751,129571770.0,129571770.0
2,1404796.0,2022-05-23,Grande São Paulo,1.0,diamond,3,467.78,7.681,144664666.0,144664666.0


In [49]:
df_comp_defects.columns

Index(['STOREKEEPER_ID', 'WEEK', 'CITY', 'LEVEL_ID', 'LEVEL_NAME', 'ORDERS',
       'GMV_TOTAL', 'COMPENSATIONS', 'DEFECT_COMPENSATIONS', 'DEFECT_ORDER'],
      dtype='object')